In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [14]:
reddit_datasets = {
    'addiction': pd.read_csv('addiction_2019_features_tfidf_256.csv'),
    'adhd': pd.read_csv('adhd_2019_features_tfidf_256.csv'),
    'alcoholism': pd.read_csv('alcoholism_2019_features_tfidf_256.csv'),
    'anxiety': pd.read_csv('anxiety_2019_features_tfidf_256.csv'),
    'jokes': pd.read_csv('jokes_2019_features_tfidf_256.csv'),
    'mentalhealth': pd.read_csv('mentalhealth_2019_features_tfidf_256.csv'),
    'conspiracy': pd.read_csv('conspiracy_2019_features_tfidf_256.csv'),
    'fitness': pd.read_csv('fitness_2019_features_tfidf_256.csv'),
    'meditation': pd.read_csv('meditation_2019_features_tfidf_256.csv'),
    'relationship': pd.read_csv('relationships_2019_features_tfidf_256.csv'),
    'teaching': pd.read_csv('teaching_2019_features_tfidf_256.csv'),
    'suicidewatch': pd.read_csv('suicidewatch_2019_features_tfidf_256.csv'),
}

for name, df in reddit_datasets.items():
    df['subreddit_name'] = name

combined_df = pd.concat(reddit_datasets.values(), ignore_index=True)


In [19]:

# Select feature columns
feature_cols = [col for col in combined_df.columns if col.startswith('tfidf_')]

X = combined_df[feature_cols]
y = combined_df['subreddit_name']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train SVM
svm = LinearSVC()
svm.fit(X_train, y_train)

# Evaluate
y_pred = svm.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.7627465948812728

Classification Report:
               precision    recall  f1-score   support

   addiction       0.61      0.49      0.54       251
        adhd       0.81      0.69      0.74      2045
  alcoholism       0.70      0.52      0.60       188
     anxiety       0.76      0.74      0.75      2643
  conspiracy       0.57      0.37      0.45       823
     fitness       0.75      0.84      0.79      2833
       jokes       0.80      0.93      0.86      5640
  meditation       0.60      0.28      0.38       541
mentalhealth       0.61      0.35      0.44      1907
relationship       0.80      0.88      0.84      4935
suicidewatch       0.72      0.78      0.75      2918
    teaching       0.56      0.12      0.19       165

    accuracy                           0.76     24889
   macro avg       0.69      0.58      0.61     24889
weighted avg       0.75      0.76      0.75     24889



In [20]:

# Twitter datasets already combined
twitter_datasets = {
    'clean_d_tweets': pd.read_csv('clean_d_tweets.csv'),
    'clean_non_d_tweets': pd.read_csv('clean_non_d_tweets.csv'),
    'd_tweets': pd.read_csv('d_tweets.csv'),
    'non_d_tweets': pd.read_csv('non_d_tweets.csv')
}

for name, df in twitter_datasets.items():
    df['twitter_name'] = name

combined_df1 = pd.concat(twitter_datasets.values(), ignore_index=True)

# Assign label
combined_df1['label'] = combined_df1['twitter_name'].apply(
    lambda x: 1 if 'd_tweets' in x else 0
)


In [22]:
# Define which subreddits are depression-related
depression_subs = [
    'addiction', 'adhd', 'alcoholism', 'anxiety',
    'mentalhealth', 'suicidewatch'
]

# Add label column
combined_df['label'] = combined_df['subreddit_name'].apply(
    lambda x: 1 if x in depression_subs else 0
)


In [23]:
reddit_text_df = combined_df[['post', 'label']]
reddit_text_df = reddit_text_df.rename(columns={'post': 'text'})

twitter_text_df = combined_df1[['tweet', 'label']]
twitter_text_df = twitter_text_df.rename(columns={'tweet': 'text'})

all_data = pd.concat([reddit_text_df, twitter_text_df], ignore_index=True)
all_data = all_data.dropna(subset=['text'])


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Split
X_train, X_test, y_train, y_test = train_test_split(
    all_data['text'],
    all_data['label'],
    test_size=0.2,
    random_state=42,
    stratify=all_data['label']
)

# Vectorize
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train SVM
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train_tfidf, y_train)

# Predict
y_pred = svm_model.predict(X_test_tfidf)

# Evaluate
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred))
print("\n🧾 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


✅ Accuracy: 0.9111356276024057

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.92      0.92     14937
           1       0.91      0.90      0.91     13162

    accuracy                           0.91     28099
   macro avg       0.91      0.91      0.91     28099
weighted avg       0.91      0.91      0.91     28099


🧾 Confusion Matrix:
 [[13702  1235]
 [ 1262 11900]]


In [33]:
new_posts = [
    "I feel so lost and hopeless lately.",
    "Just finished my morning workout, feeling great!",
    "Meditation helps me deal with stress."
]

new_tfidf = vectorizer.transform(new_posts)
preds = svm_model.predict(new_tfidf)
print(preds)  # [1, 0, 1] for example



[1 0 0]
